In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
pip install PyMuPDF
pip install pytesseract
pip install opencv-python

"""

'\npip install PyMuPDF\npip install pytesseract\n'

In [6]:
import os
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import cv2
import numpy as np
import pandas as pd

ModuleNotFoundError: No module named 'cv2'

In [ ]:
def preprocess_image_for_ocr(img):
    # Convert to grayscale
    gray_img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)

    # Binarization (thresholding) using OTSU
    _, binary_img = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Denoising
    denoised_img = cv2.medianBlur(binary_img, 3)

    return Image.fromarray(denoised_img)

In [7]:
def extract_text_from_pdf_ocr(pdf_path):
    """Extract text from a PDF file using OCR with preprocessing"""
    try:
        doc = fitz.open(pdf_path)
        ocr_text = []

        for i, page in enumerate(doc):
            # Render page to an image with high DPI (zoom=3)
            pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))
            img_bytes = pix.tobytes("png")
            pil_img = Image.open(io.BytesIO(img_bytes))

            # Preprocess the image for better OCR accuracy
            processed_img = preprocess_image_for_ocr(pil_img)

            # Perform OCR
            text = pytesseract.image_to_string(processed_img)
            if text.strip():
                ocr_text.append(text.strip())

        doc.close()
        return "\n".join(ocr_text).strip()
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return ""

In [8]:
# Define input folder containing subfolders of PDFs
input_folder = '/content/drive/MyDrive/resumes_pdf'  # Adjust this to your specific input folder path if needed

data_rows = []

# Walk through all folders and files
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith('.pdf'):
            # The direct parent folder name acts as the category header
            category = os.path.basename(root)
            pdf_path = os.path.join(root, file)

            print(f'Extracting text via OCR from: {category} --> {file}')

            # Extract text using OCR
            raw_text = extract_text_from_pdf_ocr(pdf_path)

            # Append rows (Category, Filename, Plain Text)
            data_rows.append([category, file, raw_text])

# Create a DataFrame to store and view the results
df_extracted = pd.DataFrame(data_rows, columns=['Category', 'Filename', 'Plain Text'])
display(df_extracted.head())

Extracting text via OCR from: WebDesigning --> 767f31a26171a6f8.pdf
Error reading /content/drive/MyDrive/resumes_pdf/WebDesigning/767f31a26171a6f8.pdf: name 'preprocess_image_for_ocr' is not defined
Extracting text via OCR from: WebDesigning --> 096bd41320e43e64.pdf
Error reading /content/drive/MyDrive/resumes_pdf/WebDesigning/096bd41320e43e64.pdf: name 'preprocess_image_for_ocr' is not defined
Extracting text via OCR from: WebDesigning --> 851cbb485b35f3b4.pdf
Error reading /content/drive/MyDrive/resumes_pdf/WebDesigning/851cbb485b35f3b4.pdf: name 'preprocess_image_for_ocr' is not defined
Extracting text via OCR from: WebDesigning --> 4c31ea379b8b4c66.pdf
Error reading /content/drive/MyDrive/resumes_pdf/WebDesigning/4c31ea379b8b4c66.pdf: name 'preprocess_image_for_ocr' is not defined
Extracting text via OCR from: WebDesigning --> b2c76bfd0278688f.pdf
Error reading /content/drive/MyDrive/resumes_pdf/WebDesigning/b2c76bfd0278688f.pdf: name 'preprocess_image_for_ocr' is not defined
Extra

KeyboardInterrupt: 